# Phase 2A — Parse CPLEX Full 733-Node Solution Files

## Purpose

This notebook reads the `.sol` files produced by the CPLEX Interactive Optimizer
after solving the **full 733-node p-median LP files** generated by
`phase2a_cplex_lp_full733.ipynb`.

It extracts the optimal shelter selections and block group assignments,
verifies the objective value, compares it to the PuLP/CBC benchmark, and
saves assignment CSVs for use in Phase 2B visualisation.

## Inputs Required

| File | Where it comes from |
|---|---|
| `pmedian_full_p5.sol` | CPLEX: `write pmedian_full_p5.sol` |
| `pmedian_full_p8.sol` | CPLEX: `write pmedian_full_p8.sol` |
| `pmedian_full_p10.sol` | CPLEX: `write pmedian_full_p10.sol` |
| `demand_nodes.csv` | Phase 1B output |
| `distance_matrix_network.csv` | Phase 1B output |

## Outputs

| File | Description |
|---|---|
| `cplex_full_results.json` | Objective values, gaps, selected shelters per scenario |
| `cplex_full_assignments_p5.csv` | Block group → shelter assignments for p=5 |
| `cplex_full_assignments_p8.csv` | Block group → shelter assignments for p=8 |
| `cplex_full_assignments_p10.csv` | Block group → shelter assignments for p=10 |

In [1]:
# =============================================================================
# CELL 1 — CONFIGURATION
# Update file paths to match your directory structure.
# =============================================================================

import pandas as pd
import numpy as np
import json, os, re

# ── INPUT PATHS ───────────────────────────────────────────────────────────────
DEMAND_CSV  = r"D:\GIS_Seminar_Project\CPLEX_LP_Full\demand_nodes.csv"
MATRIX_CSV  = r"D:\GIS_Seminar_Project\CPLEX_LP_Full\distance_matrix_network.csv"

# Folder where Dr. Downs saves the .sol files from CPLEX
SOL_DIR     = r"D:\GIS_Seminar_Project\CPLEX_LP_Full\Dr_Joni_Data_Solns"

# ── OUTPUT PATH ───────────────────────────────────────────────────────────────
OUTPUT_DIR  = r"D:\GIS_Seminar_Project\CPLEX_LP_Full\Dr_Joni_Data_Solns"

# ── SCENARIOS ─────────────────────────────────────────────────────────────────
P_VALUES    = [5, 8, 10]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: D:\GIS_Seminar_Project\CPLEX_LP_Full\Dr_Joni_Data_Solns


In [2]:
# =============================================================================
# CELL 2 — LOAD DATA AND DEFINE SHELTER NAMES
# =============================================================================

# ── Shelter name lookup (S0..S24) ─────────────────────────────────────────────
candidates = pd.DataFrame([
    {"SID": 0,  "NAME": "Clearwater Fundamental Middle"},
    {"SID": 1,  "NAME": "Palm Harbor Middle"},
    {"SID": 2,  "NAME": "John Hopkins Middle"},
    {"SID": 3,  "NAME": "Sexton Elementary"},
    {"SID": 4,  "NAME": "Gibbs High"},
    {"SID": 5,  "NAME": "McMullen-Booth Elementary"},
    {"SID": 6,  "NAME": "Carwise Middle"},
    {"SID": 7,  "NAME": "Jamerson Elementary"},
    {"SID": 8,  "NAME": "Sanderlin K-8"},
    {"SID": 9,  "NAME": "Ross Norton"},
    {"SID": 10, "NAME": "Campbell Park Elementary"},
    {"SID": 11, "NAME": "New Heights Elementary"},
    {"SID": 12, "NAME": "Fairmount Park Elementary"},
    {"SID": 13, "NAME": "Belleair Elementary"},
    {"SID": 14, "NAME": "Skycrest Elementary"},
    {"SID": 15, "NAME": "Largo High"},
    {"SID": 16, "NAME": "Lealman Exchange"},
    {"SID": 17, "NAME": "Mildred Helms Elementary"},
    {"SID": 18, "NAME": "Melrose Elementary"},
    {"SID": 19, "NAME": "Palm Harbor University High"},
    {"SID": 20, "NAME": "Palm Harbor University High Bldg 19"},
    {"SID": 21, "NAME": "Clearwater High"},
    {"SID": 22, "NAME": "Palm Harbor CSA"},
    {"SID": 23, "NAME": "The Coliseum"},
    {"SID": 24, "NAME": "White Chapel"},
])

# ── Load demand nodes ─────────────────────────────────────────────────────────
# Keep only the 733 block groups with non-zero population.
demand_all = pd.read_csv(DEMAND_CSV, dtype={"GEOID_JOIN": str})
demand_all["GEOID_JOIN"] = demand_all["GEOID_JOIN"].astype(str).str.strip()
demand = demand_all[demand_all["POPULATION"] > 0].reset_index(drop=True)

# ── Load travel-time matrix ───────────────────────────────────────────────────
matrix_raw = pd.read_csv(MATRIX_CSV)
matrix_raw = matrix_raw.drop(columns=["GEOID_JOIN"], errors="ignore")
matrix_raw.index = demand["GEOID_JOIN"].values
D_full = matrix_raw.values.astype(float)   # shape: (733, 25)

print(f"Demand nodes: {len(demand)}  |  Matrix shape: {D_full.shape}")
print(f"Travel-time range: {D_full.min():.2f} – {D_full.max():.2f} min")

Demand nodes: 733  |  Matrix shape: (733, 25)
Travel-time range: 0.03 – 93.80 min


In [3]:
# =============================================================================
# CELL 3 — PARSE SOLUTION FILES AND SAVE OUTPUTS
# =============================================================================

# PuLP/CBC benchmark values for comparison (from paper)
PULP_BENCHMARKS = {5: 4184071, 8: 3774238, 10: 3641580}

def parse_sol_file(sol_path):
    """
    Parse a CPLEX .sol XML file for the full 733-node problem.

    Returns
    -------
    objective : float — optimal objective value reported by CPLEX (person·min)
    selected  : list  — sorted shelter indices j where x_j = 1
    bg_asgn   : dict  — {block_group_row_index i : shelter_index j}
    """
    with open(sol_path, "r") as f:
        content = f.read()

    # Extract objective value from XML
    obj_match = re.search(r'objectiveValue="([^"]+)"', content)
    objective = float(obj_match.group(1)) if obj_match else None

    var_pattern = re.compile(r'<variable\s+name="([^"]+)"\s+[^>]*value="([^"]+)"')
    selected = []
    bg_asgn  = {}   # row index i → shelter index j

    for match in var_pattern.finditer(content):
        name  = match.group(1)
        value = float(match.group(2))
        if abs(value - 1.0) < 0.5:   # binary: treat ≥ 0.5 as 1
            if name.startswith("x_"):
                # x_{j}: shelter j is open
                j = int(name.split("_")[1])
                selected.append(j)
            elif name.startswith("y_"):
                # y_{i}_{j}: block group i assigned to shelter j
                parts = name.split("_")
                i, j  = int(parts[1]), int(parts[2])
                bg_asgn[i] = j

    return objective, sorted(selected), bg_asgn


# ── Process each scenario ─────────────────────────────────────────────────────
cplex_results = {}
print("=" * 68)
print("CPLEX FULL 733-NODE SOLUTION RESULTS")
print("=" * 68)

for p in P_VALUES:
    sol_path = os.path.join(SOL_DIR, f"pmedian_full_p{p}.sol")

    if not os.path.exists(sol_path):
        print(f"\n  WARNING: {sol_path} not found — skipping p={p}")
        continue

    print(f"\np = {p}")
    objective, selected, bg_asgn = parse_sol_file(sol_path)

    print(f"  CPLEX objective:    {objective:>15,.2f} person·min")
    print(f"  Selected shelters:")
    for j in selected:
        print(f"    ✓ S{j}: {candidates.iloc[j]['NAME']}")

    # ── Build assignment list for all 733 block groups ────────────────────────
    # For the full problem, y_i_j variables directly give assignments.
    # If CPLEX pruned any (because they were obviously 0), fall back to
    # nearest open shelter by travel time — this should rarely happen.
    demand_assignments = []
    fallback_count = 0
    for i in range(len(demand)):
        if i in bg_asgn:
            demand_assignments.append(bg_asgn[i])
        else:
            j = min(selected, key=lambda s: D_full[i, s])
            demand_assignments.append(j)
            fallback_count += 1

    if fallback_count > 0:
        print(f"  Note: {fallback_count} BGs used nearest-shelter fallback")

    # ── Verify and compare ────────────────────────────────────────────────────
    obj_verify = float(sum(
        demand["WEIGHTED_DEMAND"].iloc[i] * D_full[i, demand_assignments[i]]
        for i in range(len(demand))
    ))
    gap = (obj_verify - PULP_BENCHMARKS[p]) / PULP_BENCHMARKS[p] * 100

    print(f"  Verified objective: {obj_verify:>15,.2f} person·min")
    print(f"  PuLP/CBC benchmark: {PULP_BENCHMARKS[p]:>15,.0f} person·min")
    print(f"  Gap vs benchmark:   {gap:>+14.4f}%")

    cplex_results[p] = {
        "objective":        objective,
        "obj_verify":       round(obj_verify, 2),
        "pulp_benchmark":   PULP_BENCHMARKS[p],
        "gap_vs_pulp_pct":  round(gap, 4),
        "solve_time":       0.0,   # fill in manually from CPLEX log if needed
        "selected":         selected,
    }

    # ── Save assignment CSV ───────────────────────────────────────────────────
    cols = ["GEOID_JOIN", "LAT", "LON", "POPULATION", "WEIGHTED_DEMAND"]
    for col in ["TIER_LABEL", "MULTIPLIER"]:
        if col in demand.columns:
            cols.append(col)

    out = demand[cols].copy()
    out["METHOD"]          = "CPLEX_FULL"
    out["ASSIGNED_SID"]    = demand_assignments
    out["ASSIGNED_NAME"]   = [candidates.iloc[j]["NAME"] for j in demand_assignments]
    out["TRAVEL_TIME_MIN"] = [D_full[i, demand_assignments[i]]
                               for i in range(len(demand))]

    out_path = os.path.join(OUTPUT_DIR, f"cplex_full_assignments_p{p}.csv")
    out.to_csv(out_path, index=False)
    print(f"  Saved: cplex_full_assignments_p{p}.csv")

# ── Save summary JSON ─────────────────────────────────────────────────────────
json_path = os.path.join(OUTPUT_DIR, "cplex_full_results.json")
with open(json_path, "w") as f:
    json.dump({str(p): v for p, v in cplex_results.items()}, f, indent=2)
print(f"\nSaved: cplex_full_results.json")

print("\n" + "=" * 68)
print("Upload these outputs to Google Colab for Phase 2B figures:")
print("  cplex_full_results.json")
print("  cplex_full_assignments_p5.csv")
print("  cplex_full_assignments_p8.csv")
print("  cplex_full_assignments_p10.csv")
print("=" * 68)

CPLEX FULL 733-NODE SOLUTION RESULTS

p = 5
  CPLEX objective:       4,184,071.38 person·min
  Selected shelters:
    ✓ S1: Palm Harbor Middle
    ✓ S8: Sanderlin K-8
    ✓ S14: Skycrest Elementary
    ✓ S16: Lealman Exchange
    ✓ S17: Mildred Helms Elementary
  Verified objective:    4,184,071.38 person·min
  PuLP/CBC benchmark:       4,184,071 person·min
  Gap vs benchmark:          +0.0000%
  Saved: cplex_full_assignments_p5.csv

p = 8
  CPLEX objective:       3,774,238.41 person·min
  Selected shelters:
    ✓ S0: Clearwater Fundamental Middle
    ✓ S1: Palm Harbor Middle
    ✓ S3: Sexton Elementary
    ✓ S5: McMullen-Booth Elementary
    ✓ S8: Sanderlin K-8
    ✓ S12: Fairmount Park Elementary
    ✓ S16: Lealman Exchange
    ✓ S17: Mildred Helms Elementary
  Verified objective:    3,774,238.41 person·min
  PuLP/CBC benchmark:       3,774,238 person·min
  Gap vs benchmark:          +0.0000%
  Saved: cplex_full_assignments_p8.csv

p = 10
  CPLEX objective:       3,641,579.59 person·